<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-scientific-discovery/blob/main/notebooks/lecture-03/limits-of-deep-networks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Limits of Deep Networks

*Deep Learning for Scientific Discovery, Jagiellonian University, lecture of. Author: Marcin Płodzień.*

A trained network reaches $93\%$ test accuracy on BloodMNIST (Model E of the main notebook). This number is measured on test images that come from **the same source** as the training images. Three experiments show what it does not guarantee:

1. **Inputs from outside the training distribution.** We show the blood-cell network satellite images and random noise. It has to answer one of the eight cell types, and we measure how confident it is.
2. **A shift of the input distribution.** We change the colours of the real test cells, as a different staining or microscope would, and measure the accuracy.
3. **A shortcut.** We put a small dark square in the corner of every training image of one class, train a network, and test it on clean images and on images of other classes with the square.

All on a CPU; the trained Model E is loaded from `results/model_E.pt`.

## Setup

In [1]:
import os, io, zipfile, time
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
torch.set_num_threads(4); device = "cpu"
import os
OUT = "figures"; os.makedirs(OUT, exist_ok=True); RES = "results"
blue, orange, navy = "#005CA7", "#EB6709", "#0A2F5C"
nums = {}
def clean(ax):
    for s in ["top", "right"]: ax.spines[s].set_visible(False)

import urllib.request
def fetch(path, url):        # use the local file if it exists, otherwise download it once (e.g. in Google Colab)
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True); print("downloading", url); urllib.request.urlretrieve(url, path)
    return path
os.makedirs(RES, exist_ok=True); os.makedirs(OUT, exist_ok=True)
fetch("data/bloodmnist_64.npz", "https://zenodo.org/records/10519652/files/bloodmnist_64.npz?download=1")
fetch("data/bloodmnist.npz", "https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1")
fetch("data/EuroSAT.zip", "https://huggingface.co/datasets/torchgeo/eurosat/resolve/main/EuroSAT.zip")
fetch(f"{RES}/model_E.pt", "https://raw.githubusercontent.com/MarcinPlodzien/deep-learning-for-scientific-discovery/main/notebooks/lecture-03/results/model_E.pt")   # Model E of the blood-cell notebook

class_names = ["basophil", "eosinophil", "erythroblast", "immature granulocyte", "lymphocyte", "monocyte", "neutrophil", "platelet"]
data = np.load("data/bloodmnist_64.npz")
X_train, X_test, y_test = data["train_images"], data["test_images"], data["test_labels"].ravel()
mean = X_train.reshape(-1, 3).mean(axis=0) / 255; std = X_train.reshape(-1, 3).std(axis=0) / 255   # as in the main notebook
def to_tensor(X):                                       # uint8 images (n, 64, 64, 3) -> normalised tensor (n, 3, 64, 64)
    x = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.tensor(mean, dtype=torch.float32)[:, None, None]) / torch.tensor(std, dtype=torch.float32)[:, None, None]

def conv_block(n_in, n_out):
    return nn.Sequential(nn.Conv2d(n_in, n_out, 3, padding=1), nn.BatchNorm2d(n_out), nn.ReLU(),
                         nn.Conv2d(n_out, n_out, 3, padding=1), nn.BatchNorm2d(n_out), nn.ReLU())
class GapCNN(nn.Module):                                # Model E of the main notebook
    def __init__(self, n_channels=3, n_classes=8):
        super().__init__()
        self.features = nn.Sequential(conv_block(n_channels, 32), nn.MaxPool2d(2), conv_block(32, 64), nn.MaxPool2d(2), conv_block(64, 128))
        self.fc = nn.Linear(128, n_classes)
    def forward(self, x): return self.fc(self.features(x).mean(dim=(2, 3)))
model_E = GapCNN(); model_E.load_state_dict(torch.load(f"{RES}/model_E.pt", map_location="cpu")); model_E.eval()

def probabilities(model, X):
    with torch.no_grad():
        return torch.cat([model(X[k:k + 500]).softmax(1) for k in range(0, len(X), 500)]).numpy()
p_test = probabilities(model_E, to_tensor(X_test))
nums["E_test_acc"] = float((p_test.argmax(1) == y_test).mean())
print(f"Model E, test accuracy on real blood cells: {nums['E_test_acc']:.3f}")

Model E, test accuracy on real blood cells: 0.932


## 1. Inputs from outside the training distribution

The network ends with a softmax over the eight cell types: whatever the input, the eight probabilities sum to 1, and one class gets the largest. There is no answer "this is not a blood cell". We give it

* **satellite images** (EuroSAT, $64\times64$ RGB tiles of Sentinel-2 images: forests, rivers, highways, ...), the data set of the second part of the main notebook;
* **random noise**: every pixel drawn uniformly from $0$ to $255$.

The **confidence** of a prediction is the largest of the eight output probabilities $q_1,\dots,q_8$ of the softmax, $\max_k q_k$.

In [2]:
archive = zipfile.ZipFile("data/EuroSAT.zip")
files = sorted(n for n in archive.namelist() if n.lower().endswith(".jpg"))
rng = np.random.default_rng(1)
pick = rng.choice(len(files), 1000, replace=False)
S = np.stack([np.array(Image.open(io.BytesIO(archive.read(files[i]))).convert("RGB")) for i in pick])
N = rng.integers(0, 256, size=(1000, 64, 64, 3), dtype=np.uint8)
p_sat, p_noise = probabilities(model_E, to_tensor(S)), probabilities(model_E, to_tensor(N))
conf = {"real test cells": p_test.max(1), "satellite images": p_sat.max(1), "random noise": p_noise.max(1)}
for k, v in conf.items():
    nums[f"conf_median_{k}"] = float(np.median(v)); nums[f"conf_above_09_{k}"] = float((v > 0.9).mean())
    print(f"{k:18s}: median confidence {np.median(v):.3f}; fraction with confidence > 0.9: {(v > 0.9).mean():.3f}")
print("satellite images are called:", {class_names[c]: int(n) for c, n in enumerate(np.bincount(p_sat.argmax(1), minlength=8)) if n})
print("noise images are called:", {class_names[c]: int(n) for c, n in enumerate(np.bincount(p_noise.argmax(1), minlength=8)) if n})

fig = plt.figure(figsize=(11, 3.6))
ax = fig.add_axes([0.0, 0.12, 0.42, 0.78])
bins = np.linspace(0.1, 1, 37)
for (k, v), col in zip(conf.items(), [blue, orange, "grey"]):
    ax.hist(v, bins=bins, density=True, histtype="step", lw=2.2, color=col, label=k)
ax.set_xlabel("confidence = largest of the 8 probabilities"); ax.set_ylabel("density (log scale)"); ax.set_yscale("log"); ax.legend(frameon=False, fontsize=9, loc="upper left"); clean(ax)
ax.set_title("the blood-cell network on three kinds of input", fontsize=11)
show = list(np.argsort(-p_sat.max(1))[:4])
for j, i in enumerate(show):
    a = fig.add_axes([0.47 + 0.135 * j, 0.42, 0.12, 0.48]); a.imshow(S[i]); a.axis("off")
    a.set_title(f"{class_names[p_sat[i].argmax()]}\n{p_sat[i].max():.3f}", fontsize=9)
a = fig.add_axes([0.47, 0.0, 0.12, 0.36]); a.imshow(N[0]); a.axis("off")
a.set_title(f"{class_names[p_noise[0].argmax()]}  {p_noise[0].max():.3f}", fontsize=9)
fig.text(0.61, 0.17, "top: the four most confident satellite tiles\nleft: one noise image\n(predicted cell type, confidence)", fontsize=9)
plt.savefig(f"{OUT}/limits_ood_confidence.png", dpi=170, bbox_inches="tight"); plt.show()

real test cells   : median confidence 0.998; fraction with confidence > 0.9: 0.857
satellite images  : median confidence 1.000; fraction with confidence > 0.9: 0.993
random noise      : median confidence 1.000; fraction with confidence > 0.9: 1.000
satellite images are called: {'basophil': 5, 'eosinophil': 1, 'erythroblast': 991, 'lymphocyte': 2, 'platelet': 1}
noise images are called: {'basophil': 1000}


/tmp/ipykernel_3908662/2110671245.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/limits_ood_confidence.png", dpi=170, bbox_inches="tight"); plt.show()


The network has never seen a satellite image or noise, yet it assigns them cell types, part of them with high confidence. A high softmax probability measures how well an input matches one class *compared with the other seven*, not whether the input looks like the training data at all.

## 2. A Different Staining

Blood smears are stained, and the colour depends on the stain batch, the protocol and the microscope. We imitate this by scaling the three colour channels of the real test images: red by $1-s$, green unchanged, blue by $1+s$, for a shift $s$ from $0$ to $0.4$, and by a global brightness factor. The cells are the same; only the colours change.

In [3]:
def stain(X, s):            # scale red by (1 - s) and blue by (1 + s); X: uint8 (n, 64, 64, 3)
    g = np.array([1 - s, 1.0, 1 + s])
    return np.clip(X.astype(np.float32) * g, 0, 255).astype(np.uint8)
def brightness(X, b):
    return np.clip(X.astype(np.float32) * b, 0, 255).astype(np.uint8)
shifts = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]; brights = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
acc_s = [float((probabilities(model_E, to_tensor(stain(X_test, s))).argmax(1) == y_test).mean()) for s in shifts]
acc_b = [float((probabilities(model_E, to_tensor(brightness(X_test, b))).argmax(1) == y_test).mean()) for b in brights]
nums["stain_shift"], nums["stain_acc"] = shifts, [round(a, 3) for a in acc_s]
nums["brightness"], nums["brightness_acc"] = brights, [round(a, 3) for a in acc_b]
print("colour shift s:", shifts); print("accuracy      :", np.round(acc_s, 3))
print("brightness b  :", brights); print("accuracy      :", np.round(acc_b, 3))

fig = plt.figure(figsize=(11, 3.4))
for j, s in enumerate([0.0, 0.1, 0.2, 0.4]):
    a = fig.add_axes([0.0 + 0.105 * j, 0.45, 0.095, 0.45]); a.imshow(stain(X_test[:1], s)[0]); a.axis("off"); a.set_title(f"s = {s}", fontsize=10)
fig.text(0.0, 0.3, "the same test cell (" + class_names[y_test[0]] + "),\nred scaled by 1 - s, blue by 1 + s", fontsize=9)
ax = fig.add_axes([0.5, 0.12, 0.22, 0.78]); ax.plot(shifts, acc_s, "o-", color=orange, lw=2); ax.set_ylim(0, 1)
ax.set_xlabel("colour shift $s$"); ax.set_ylabel("test accuracy"); ax.axhline(1 / 8, color="grey", ls=":", lw=1); clean(ax); ax.set_title("staining", fontsize=11)
ax2 = fig.add_axes([0.78, 0.12, 0.22, 0.78]); ax2.plot(brights, acc_b, "o-", color=blue, lw=2); ax2.set_ylim(0, 1)
ax2.set_xlabel("brightness factor"); ax2.axhline(1 / 8, color="grey", ls=":", lw=1); clean(ax2); ax2.set_title("brightness", fontsize=11)
plt.savefig(f"{OUT}/limits_stain_shift.png", dpi=170, bbox_inches="tight"); plt.show()

colour shift s: [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]
accuracy      : [0.932 0.866 0.647 0.29  0.108 0.028 0.072]
brightness b  : [0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
accuracy      : [0.093 0.166 0.654 0.932 0.761 0.348]


/tmp/ipykernel_3908662/2223290882.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/limits_stain_shift.png", dpi=170, bbox_inches="tight"); plt.show()


A colour change that a haematologist would hardly notice moves the images away from the training distribution, and the accuracy falls (dotted line: guessing among 8 classes). The test accuracy of the main notebook was measured on images of the same staining as the training images; it says nothing about another laboratory.

## 3. A shortcut

A network uses whatever separates the training classes. Suppose that, in the training set, every **platelet** image carries a small dark square in the top-left corner: a marker that, for instance, one laboratory adds to its images, and that laboratory happens to supply all the platelets. We train a small CNN on these images ($28\times28$ pixels, for speed) and test it

* on clean test images: are the platelets still recognised?
* on test images of the other seven types **with** the square: are they called platelets?

For comparison, the same network is trained on clean images.

In [4]:
d28 = np.load("data/bloodmnist.npz")
A_tr, a_tr = d28["train_images"], d28["train_labels"].ravel(); A_te, a_te = d28["test_images"], d28["test_labels"].ravel()
PLT = class_names.index("platelet")
def mark(X):                # a dark 4 x 4 square in the top-left corner
    X = X.copy(); X[:, 1:5, 1:5, :] = 20; return X
m28 = A_tr.reshape(-1, 3).mean(0) / 255; s28 = A_tr.reshape(-1, 3).std(0) / 255
def t28(X):
    x = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.tensor(m28, dtype=torch.float32)[:, None, None]) / torch.tensor(s28, dtype=torch.float32)[:, None, None]
def small_cnn():
    return nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 8))
def fit(X, y, epochs=6, seed=0):
    torch.manual_seed(seed); m = small_cnn(); opt = torch.optim.Adam(m.parameters(), 1e-3); X = t28(X); y = torch.tensor(y, dtype=torch.long)
    for ep in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(perm), 128):
            j = perm[i:i + 128]; loss = F.cross_entropy(m(X[j]), y[j]); opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()
A_tr_marked = A_tr.copy(); A_tr_marked[a_tr == PLT] = mark(A_tr[a_tr == PLT])
t0 = time.time()
net_clean, net_short = fit(A_tr, a_tr), fit(A_tr_marked, a_tr)
torch.save(net_short.state_dict(), f"{RES}/limits_shortcut_cnn.pt"); torch.save(net_clean.state_dict(), f"{RES}/limits_clean_cnn.pt")
print(f"two networks trained in {time.time() - t0:.0f} s")
def pred(m, X):
    with torch.no_grad(): return m(t28(X)).argmax(1).numpy()
others = a_te != PLT
res = {}
for name, m in [("trained on clean images", net_clean), ("trained with the marker", net_short)]:
    p_clean = pred(m, A_te); p_mark = pred(m, mark(A_te))
    res[name] = dict(acc=float((p_clean == a_te).mean()), platelet_recall=float((p_clean[a_te == PLT] == PLT).mean()),
                     others_called_platelet=float((p_mark[others] == PLT).mean()))
    print(f"{name:26s}: clean test accuracy {res[name]['acc']:.3f}; platelets recognised (clean) {res[name]['platelet_recall']:.3f}; "
          f"other cells WITH marker called platelet {res[name]['others_called_platelet']:.3f}")
nums["shortcut"] = res

fig = plt.figure(figsize=(11, 3.2))
ex = [np.flatnonzero(a_tr == PLT)[0], np.flatnonzero(a_te == class_names.index("neutrophil"))[0]]
a = fig.add_axes([0.0, 0.25, 0.14, 0.62]); a.imshow(A_tr_marked[ex[0]]); a.axis("off"); a.set_title("training platelet\nwith marker", fontsize=9)
a = fig.add_axes([0.16, 0.25, 0.14, 0.62]); a.imshow(mark(A_te[ex[1]:ex[1] + 1])[0]); a.axis("off"); a.set_title("test neutrophil\nwith marker", fontsize=9)
ax = fig.add_axes([0.38, 0.15, 0.6, 0.72])
labels = ["clean test accuracy", "platelets recognised\n(clean test images)", "other cells with marker\ncalled platelet"]
keys = ["acc", "platelet_recall", "others_called_platelet"]; x = np.arange(3)
for k, (name, col) in enumerate([("trained on clean images", blue), ("trained with the marker", orange)]):
    v = [res[name][q] for q in keys]; ax.bar(x + (k - 0.5) * 0.38, v, 0.38, color=col, label=name)
    for xi, vi in zip(x + (k - 0.5) * 0.38, v): ax.text(xi, vi + 0.02, f"{vi:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9); ax.set_ylim(0, 1.35); ax.legend(frameon=False, fontsize=9, loc="upper right", ncol=2); clean(ax)
plt.savefig(f"{OUT}/limits_shortcut.png", dpi=170, bbox_inches="tight"); plt.show()

two networks trained in 93 s


trained on clean images   : clean test accuracy 0.906; platelets recognised (clean) 1.000; other cells WITH marker called platelet 0.003


trained with the marker   : clean test accuracy 0.764; platelets recognised (clean) 0.000; other cells WITH marker called platelet 0.312


/tmp/ipykernel_3908662/2496564787.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(f"{OUT}/limits_shortcut.png", dpi=170, bbox_inches="tight"); plt.show()


The network trained with the marker learns the marker. A cell of any type carrying the square is called a platelet, while the test accuracy on clean images alone would not reveal it. In real data such shortcuts are a hospital's text label on an X-ray, a scale bar, the background of the photograph: anything that happens to correlate with the class in the training set. Class activation maps, and test data from an independent source, are the tools that reveal them.

**Try it.** Make the marker smaller (2 × 2 pixels) or put it on only half of the platelet images. Is the shortcut still learned?

## Numbers used on the slides

In [5]:
import json; json.dump(nums, open(f"{RES}/numbers_limits.json", "w"), indent=1); print(json.dumps(nums, indent=1))

{
 "E_test_acc": 0.9318912598655364,
 "conf_median_real test cells": 0.9978804588317871,
 "conf_above_09_real test cells": 0.8573516515638702,
 "conf_median_satellite images": 1.0,
 "conf_above_09_satellite images": 0.993,
 "conf_median_random noise": 1.0,
 "conf_above_09_random noise": 1.0,
 "stain_shift": [
  0.0,
  0.05,
  0.1,
  0.15,
  0.2,
  0.3,
  0.4
 ],
 "stain_acc": [
  0.932,
  0.866,
  0.647,
  0.29,
  0.108,
  0.028,
  0.072
 ],
 "brightness": [
  0.7,
  0.8,
  0.9,
  1.0,
  1.1,
  1.2
 ],
 "brightness_acc": [
  0.093,
  0.166,
  0.654,
  0.932,
  0.761,
  0.348
 ],
 "shortcut": {
  "trained on clean images": {
   "acc": 0.9064600993861444,
   "platelet_recall": 1.0,
   "others_called_platelet": 0.0027109454422229754
  },
  "trained with the marker": {
   "acc": 0.7643963753288512,
   "platelet_recall": 0.0,
   "others_called_platelet": 0.3117587258556422
  }
 }
}
